# Mindedness index and state maps

Standalone program. Reads every offense CSV in `DATA_DIR` (columns `law_number`, `conduct_merged`, `result_merged`, `circumstance_merged`), gives every statute a single **mindedness** score, averages those scores by jurisdiction, and maps the results. A coded value of `n/a` is treated as **Silent**.

**Mindedness (one number per statute).** Each element's coded mens rea gets a weight: Purposive 1.00, Knowing 0.75, Reckless 0.50, Negligent 0.25, Strict Liability 0, Silent 0. A statute's mindedness is the average weight across its elements, scaled to 0–100:

$$M_{s} = 100 \times \frac{1}{|E|}\sum_{e \in E} w(\text{level}_{s,e}), \qquad E = \{\text{conduct, result, circumstance}\}$$

100 means purpose is required for every element; 0 means nothing is stated (or strict liability is expressly imposed). With Silent weighted 0, the index equals **coverage × intensity**: the share of elements with a stated mens rea times the average weight of the ones that are stated.

**Jurisdiction score.** The mean of that jurisdiction's statute scores. Every jurisdiction is scored on the same set of offenses, so offense-level quirks (e.g. DUI having no result element) shift every jurisdiction equally and don't affect the ranking.

**Jurisdiction key.** `law_number` runs 1–51 alphabetically, with DC alphabetised as "Washington, D.C.": 47 = Washington, 48 = DC, 49 = West Virginia, 50 = Wisconsin, 51 = Wyoming. The key is the `JURISDICTIONS` list in the first code cell.

**Outputs** (in a `figures` folder next to this notebook): `mindedness_map` (one map, mean score per jurisdiction) and `mindedness_by_offense_maps` (one small map per offense), each as PNG and SVG; plus `mindedness_by_state.csv` and `mindedness_by_statute.csv`.

**Scales automatically.** Offense and jurisdiction counts come from the data; the per-offense page picks its own grid and text size, like the bar graphs.

**To run:** put the CSVs in the same folder as this notebook (or set `DATA_DIR`), then *Run All*. The first run downloads U.S. Census state boundaries (internet needed once) and caches them as `us_states_51.geojson`.

Requires `pandas`, `numpy`, `matplotlib`, `geopandas` (`pip install geopandas`) and `requests`.

In [ ]:
import re
import tempfile
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_rgb, to_rgba
from matplotlib.patches import Rectangle
from IPython.display import display, Markdown

%matplotlib inline
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

# ---- input / output -----------------------------------------------------------------
DATA_DIR = Path(".")                               # folder containing the offense CSVs
FILE_GLOB = "*_rounded_down.csv"                   # which files count as offense files
NAME_PATTERN = r"-sol_(.+)_rounded_down\.csv$"     # offense name = the part captured in ( )
OUT_DIR = Path("figures")                          # output folder, created next to this notebook
OUT_DIR.mkdir(exist_ok=True)

# ---- optional sanity checks (None = accept whatever is in the folder) ----------------
EXPECTED_OFFENSES = None          # e.g. 50 -> stop with an error if a different number is read
EXPECTED_JURISDICTIONS = None     # e.g. 51

# ---- coding scheme --------------------------------------------------------------------
ELEMENTS = {
    "conduct_merged": "Conduct",
    "result_merged": "Result",
    "circumstance_merged": "Circumstance",
}
EL_LABELS = list(ELEMENTS.values())
LEVELS = ["Purposive", "Knowing", "Reckless", "Negligent", "Strict Liability", "Silent"]

# ---- mindedness weights ----------------------------------------------------------------
MINDEDNESS_WEIGHTS = {"Purposive": 1.00, "Knowing": 0.75, "Reckless": 0.50,
                      "Negligent": 0.25, "Strict Liability": 0.00, "Silent": 0.00}
# e.g. set "Silent": 0.50 to read silence as a recklessness default (cf. MPC 2.02(3))

# ---- jurisdiction key: position in this list = law_number (1, 2, 3, ...) ----------------
JURISDICTIONS = [
    ("Alabama", "AL"), ("Alaska", "AK"), ("Arizona", "AZ"), ("Arkansas", "AR"), ("California", "CA"),
    ("Colorado", "CO"), ("Connecticut", "CT"), ("Delaware", "DE"), ("Florida", "FL"), ("Georgia", "GA"),
    ("Hawaii", "HI"), ("Idaho", "ID"), ("Illinois", "IL"), ("Indiana", "IN"), ("Iowa", "IA"),
    ("Kansas", "KS"), ("Kentucky", "KY"), ("Louisiana", "LA"), ("Maine", "ME"), ("Maryland", "MD"),
    ("Massachusetts", "MA"), ("Michigan", "MI"), ("Minnesota", "MN"), ("Mississippi", "MS"), ("Missouri", "MO"),
    ("Montana", "MT"), ("Nebraska", "NE"), ("Nevada", "NV"), ("New Hampshire", "NH"), ("New Jersey", "NJ"),
    ("New Mexico", "NM"), ("New York", "NY"), ("North Carolina", "NC"), ("North Dakota", "ND"), ("Ohio", "OH"),
    ("Oklahoma", "OK"), ("Oregon", "OR"), ("Pennsylvania", "PA"), ("Rhode Island", "RI"), ("South Carolina", "SC"),
    ("South Dakota", "SD"), ("Tennessee", "TN"), ("Texas", "TX"), ("Utah", "UT"), ("Vermont", "VT"),
    ("Virginia", "VA"), ("Washington", "WA"),
    ("District of Columbia", "DC"),   # 48: alphabetised as "Washington, D.C."
    ("West Virginia", "WV"), ("Wisconsin", "WI"), ("Wyoming", "WY"),
]

# ---- map appearance ---------------------------------------------------------------------
MINDEDNESS_CMAP = LinearSegmentedColormap.from_list(
    "mindedness", ["#F7E8EE", "#F9C2D5", "#F27DA8", "#E6005D", "#B8004A", "#7A0030"])
FONT = 7                   # base text size (pt)

# Per-offense page (small multiples)
PAGE_SIZE = (8.5, 11)      # inches; portrait letter
PANELS_PER_PAGE = None     # None = every offense on one page; e.g. 24 = pages of 24
NCOLS = None               # None = choose columns automatically; or force a number, e.g. 4
MIN_FONT = 4.5             # panel titles never shrink below this as offenses are added
MAX_FONT = 9               # ...or grow above this when there are only a few offenses

# ---- state boundaries ----------------------------------------------------------------------
BOUNDARY_CACHE = Path("us_states_51.geojson")   # written on first run; delete it to re-download
BOUNDARY_SOURCES = [
    # U.S. Census Bureau cartographic boundary file (1:20,000,000)
    "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_20m.zip",
    # fallback: public GeoJSON derived from Census boundaries
    "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
]


In [ ]:
SMALL_WORDS = {"a", "an", "the", "to", "with", "of", "and", "or", "in", "on", "for"}

def display_name(stem: str) -> str:
    """'Driving_with_a_Suspended_License' -> 'Driving with a Suspended License',
    'Kidnapping_1st_degree' -> 'Kidnapping 1st Degree', 'DUI' stays 'DUI'."""
    words = stem.replace("_", " ").split()
    return " ".join(
        w.lower() if (i > 0 and w.lower() in SMALL_WORDS) else w[0].upper() + w[1:]
        for i, w in enumerate(words)
    )

# ---- read every matching file ----------------------------------------------------------
frames, skipped = [], []
for path in sorted(DATA_DIR.glob(FILE_GLOB)):
    m = re.search(NAME_PATTERN, path.name)
    if not m:
        skipped.append(path.name)
        continue
    # keep_default_na=False keeps the literal string "n/a" instead of turning it into NaN
    df = pd.read_csv(path, keep_default_na=False)
    missing_cols = {"law_number", *ELEMENTS} - set(df.columns)
    if missing_cols:
        raise ValueError(f"{path.name} is missing columns: {sorted(missing_cols)}")
    df["offense"] = display_name(m.group(1))
    df["source_file"] = path.name
    frames.append(df)

if skipped:
    print(f"WARNING: skipped {len(skipped)} file(s) whose names don't match NAME_PATTERN:\n  "
          + "\n  ".join(skipped))
if not frames:
    raise FileNotFoundError(f"No offense files matching {FILE_GLOB!r} in {DATA_DIR.resolve()}")

raw = pd.concat(frames, ignore_index=True)

# ---- consistency checks (all derived from the data, nothing hard-coded) ---------------
clash = raw.groupby("offense")["source_file"].unique().loc[lambda s: s.map(len) > 1]
if not clash.empty:
    raise ValueError("Different files produce the same offense name:\n"
                     + "\n".join(f"  {o}: {list(f)}" for o, f in clash.items()))

dupe_rows = raw.groupby("offense")["law_number"].agg(lambda s: int(s.duplicated().sum())).loc[lambda s: s > 0]
if not dupe_rows.empty:
    raise ValueError(f"Repeated law_number rows within an offense:\n{dupe_rows.to_string()}")

ALL_JURIS = sorted(raw["law_number"].unique())
N_JURIS = len(ALL_JURIS)
coverage = raw.groupby("offense")["law_number"].nunique()
incomplete = coverage[coverage != N_JURIS]
if not incomplete.empty:
    raise ValueError(f"These offenses don't cover all {N_JURIS} jurisdictions "
                     f"(rows found shown):\n{incomplete.to_string()}")

N_OFFENSES = raw["offense"].nunique()
if EXPECTED_OFFENSES is not None and N_OFFENSES != EXPECTED_OFFENSES:
    raise ValueError(f"Expected {EXPECTED_OFFENSES} offenses, found {N_OFFENSES}")
if EXPECTED_JURISDICTIONS is not None and N_JURIS != EXPECTED_JURISDICTIONS:
    raise ValueError(f"Expected {EXPECTED_JURISDICTIONS} jurisdictions, found {N_JURIS}")

long = (
    raw.melt(id_vars=["offense", "law_number"], value_vars=list(ELEMENTS),
             var_name="element", value_name="mens_rea")
       .assign(element=lambda d: d["element"].map(ELEMENTS),
               mens_rea=lambda d: d["mens_rea"].astype(str).str.strip().replace({"n/a": "Silent"}))
)
long["stated"] = long["mens_rea"] != "Silent"

unknown = set(long["mens_rea"]) - set(LEVELS)
if unknown:
    raise ValueError(f"Unexpected mens rea labels: {sorted(unknown)}")

print(f"OK: {N_OFFENSES} offenses x {N_JURIS} jurisdictions x {len(ELEMENTS)} elements "
      f"= {len(long):,} coded element-cells")

all_silent = long.groupby("offense")["stated"].sum().loc[lambda s: s == 0].index.tolist()
if all_silent:
    print(f"WARNING: no stated mens rea anywhere in: {', '.join(all_silent)} "
          "(check whether this is real silence or an extraction problem)")


In [ ]:
def show(df, caption):
    display(Markdown(f"**{caption}**"))
    display(df)

# ---- jurisdiction key ------------------------------------------------------------------
LAW_TO_ABBR = {i + 1: abbr for i, (_, abbr) in enumerate(JURISDICTIONS)}
ABBR_TO_NAME = {abbr: name for name, abbr in JURISDICTIONS}
unmapped = sorted(set(ALL_JURIS) - set(LAW_TO_ABBR))
if unmapped:
    raise ValueError(f"law_number(s) {unmapped} have no entry in JURISDICTIONS; add them to the key")
not_in_data = [a for n, a in LAW_TO_ABBR.items() if n not in set(ALL_JURIS)]
if not_in_data:
    print(f"Note: no data for {', '.join(not_in_data)}; shown blank (white) on the maps")

# Offense order: by amount of stated mens rea data (same order as the bar graphs)
order = long.groupby("offense")["stated"].sum().sort_values(ascending=False, kind="stable").index.tolist()

long["abbr"] = long["law_number"].map(LAW_TO_ABBR)
long["weight"] = long["mens_rea"].map(MINDEDNESS_WEIGHTS)

# One score per statute (offense x jurisdiction)
statute_m = (long.groupby(["offense", "abbr"])
                 .agg(mindedness=("weight", "mean"), coverage=("stated", "mean"))
                 .assign(mindedness=lambda d: d["mindedness"] * 100, coverage=lambda d: d["coverage"] * 100))

# One score per jurisdiction (mean of its statutes)
state_m = (statute_m.groupby("abbr")
                    .agg(mean_mindedness=("mindedness", "mean"),
                         median_mindedness=("mindedness", "median"),
                         coverage_pct=("coverage", "mean"),
                         zero_statutes=("mindedness", lambda s: int((s == 0).sum()))))
state_m["intensity"] = state_m["mean_mindedness"] / state_m["coverage_pct"]   # avg weight of stated elements (0-1)
state_m.insert(0, "jurisdiction", state_m.index.map(ABBR_TO_NAME))
state_m = state_m.sort_values("mean_mindedness", ascending=False)
state_m.insert(0, "rank", range(1, len(state_m) + 1))

# Save
matrix = statute_m["mindedness"].unstack("offense").reindex(columns=order)
matrix.index = [f"{ABBR_TO_NAME[a]} ({a})" for a in matrix.index]
for name, df, label in [("mindedness_by_statute.csv", matrix.round(1), "jurisdiction"),
                        ("mindedness_by_state.csv", state_m.round(2), "abbr")]:
    df.to_csv(OUT_DIR / name, index_label=label)
    print(f"Saved {(OUT_DIR / name).resolve()}")

sm = statute_m["mindedness"]
display(Markdown(
    f"**Statute level (n = {len(sm):,}):** mean {sm.mean():.1f}, median {sm.median():.1f}, "
    f"SD {sm.std():.1f}; {int((sm == 0).sum())} statutes score 0 and {int((sm == 100).sum())} score 100.  \n"
    f"**Jurisdiction level (n = {len(state_m)}):** mean {state_m['mean_mindedness'].mean():.1f}, "
    f"range {state_m['mean_mindedness'].min():.1f}\u2013{state_m['mean_mindedness'].max():.1f}, "
    f"SD {state_m['mean_mindedness'].std():.1f}."
))
show(state_m.round(2), "Mindedness by jurisdiction (ranked)")


In [ ]:
import requests
import geopandas as gpd
from shapely.ops import polylabel

def load_state_boundaries():
    if BOUNDARY_CACHE.exists():
        return gpd.read_file(BOUNDARY_CACHE)
    errors = []
    for url in BOUNDARY_SOURCES:
        try:
            resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
            resp.raise_for_status()
            with tempfile.NamedTemporaryFile(suffix=".zip" if url.endswith(".zip") else ".json",
                                             delete=False) as tmp:
                tmp.write(resp.content)
            gdf = gpd.read_file(tmp.name)
            print(f"Boundaries loaded from {url}")
            break
        except Exception as e:
            errors.append(f"{url}: {e}")
    else:
        raise RuntimeError("Could not download state boundaries:\n" + "\n".join(errors))

    name_col = next(c for c in gdf.columns if c.lower() == "name")
    name_to_abbr = {name: abbr for name, abbr in JURISDICTIONS}
    gdf = gdf[gdf[name_col].isin(name_to_abbr)].copy()
    gdf["abbr"] = gdf[name_col].map(name_to_abbr)
    gdf = gdf[["abbr", "geometry"]].to_crs("EPSG:4326")
    gdf.to_file(BOUNDARY_CACHE, driver="GeoJSON")
    return gdf

states_geo = load_state_boundaries()
no_shape = sorted(set(state_m.index) - set(states_geo["abbr"]))
if no_shape:
    print(f"Note: no map shape for {', '.join(no_shape)}; included in the CSVs but not drawn on the maps")

# Trim far-flung islands (western Aleutians, NW Hawaiian Islands) so the insets stay compact
for abbr, bbox in {"AK": (-180, 50, -129, 72), "HI": (-161, 18.5, -154.5, 22.5)}.items():
    m = states_geo["abbr"] == abbr
    states_geo.loc[m, "geometry"] = states_geo.loc[m, "geometry"].clip_by_rect(*bbox)

# Equal-area projections
GEO = {
    "lower48": states_geo[~states_geo["abbr"].isin(["AK", "HI"])].to_crs("EPSG:5070"),  # Conus Albers
    "AK": states_geo[states_geo["abbr"] == "AK"].to_crs("EPSG:3338"),                   # Alaska Albers
    "HI": states_geo[states_geo["abbr"] == "HI"].to_crs("ESRI:102007"),                 # Hawaii Albers
}

def label_point(geom):
    # Visual center of the largest polygon (keeps labels off panhandles and islands)
    if geom.geom_type == "MultiPolygon":
        geom = max(geom.geoms, key=lambda g: g.area)
    return polylabel(geom, tolerance=max(geom.length / 500, 1))

LABEL_PTS = {key: {a: label_point(g) for a, g in zip(df["abbr"], df.geometry)} for key, df in GEO.items()}


In [ ]:
CALLOUTS = ["MA", "RI", "CT", "NJ", "DE", "MD", "DC"]   # too small to label in place
LABEL_OUTSIDE = {"HI": (10, 14)}                         # label beside the shape (offset in points)

def _text_color(rgba):
    r, g, b = to_rgb(rgba)
    return "white" if (0.299 * r + 0.587 * g + 0.114 * b) < 0.55 else "black"

def draw_us_map(ax, values, norm, cmap=MINDEDNESS_CMAP, labels=True, callouts=CALLOUTS,
                abbr_size=7, value_size=6, edge="white", edge_lw=0.5, callout_size=6.5):
    # Choropleth of the states + DC on `ax`, with Alaska/Hawaii insets and coastal callouts.
    has = lambda a: pd.notna(values.get(a, np.nan))
    color_of = lambda a: cmap(norm(values[a])) if has(a) else to_rgba("#FFFFFF")   # no data -> white
    value_text = lambda a: f"{values[a]:.1f}" if has(a) else "\u2014"

    def paint(target, key):
        df = GEO[key]
        df.plot(ax=target, color=[color_of(a) for a in df["abbr"]], edgecolor=edge if edge else "none",
                linewidth=edge_lw)
        target.set_aspect("equal")
        target.axis("off")
        if not labels:
            return
        for a, pt in LABEL_PTS[key].items():
            if a in callouts:
                continue
            dx, dy = LABEL_OUTSIDE.get(a, (0, 0))
            tc = "black" if a in LABEL_OUTSIDE else _text_color(color_of(a))
            ha = "left" if a in LABEL_OUTSIDE else "center"
            target.annotate(a, (pt.x, pt.y), xytext=(dx, dy + value_size * 0.55), textcoords="offset points",
                            ha=ha, va="center", fontsize=abbr_size, fontweight="bold", color=tc)
            target.annotate(value_text(a), (pt.x, pt.y), xytext=(dx, dy - abbr_size * 0.6),
                            textcoords="offset points", ha=ha, va="center", fontsize=value_size, color=tc)

    paint(ax, "lower48")
    xmin, ymin, xmax, ymax = GEO["lower48"].total_bounds
    w, h = xmax - xmin, ymax - ymin
    ax.set_xlim(xmin - 0.01 * w, xmax + (0.14 if callouts else 0.01) * w)
    ax.set_ylim(ymin - 0.02 * h, ymax + 0.02 * h)

    # Alaska and Hawaii insets in the empty lower-left corner
    for key, rect in [("AK", [0.0, 0.0, 0.2, 0.28]), ("HI", [0.2, 0.0, 0.11, 0.13])]:
        if len(GEO[key]):
            paint(ax.inset_axes(rect), key)

    # Coastal callouts: color swatch + label, stacked by latitude with leader lines
    if callouts:
        sw = 0.03 * h                                  # swatch size (data units)
        spacing = sw * 1.35
        x_call = xmax + 0.02 * w
        pts = LABEL_PTS["lower48"]
        slot = np.inf
        for y, a in sorted(((pts[a].y, a) for a in callouts if a in pts), reverse=True):
            slot = min(y, slot - spacing)
            ax.plot([pts[a].x, x_call - 0.15 * sw], [pts[a].y, slot], color="#8C8C8C", lw=0.4, zorder=3)
            ax.add_patch(Rectangle((x_call, slot - sw / 2), sw, sw, facecolor=color_of(a),
                                   edgecolor="#8C8C8C", linewidth=0.3, zorder=3))
            label = f"{a}  {value_text(a)}" if labels else a
            ax.text(x_call + 1.5 * sw, slot, label, va="center", fontsize=callout_size,
                    fontweight="bold" if labels else "normal")

def nice_range(vals, step=5):
    return np.floor(np.nanmin(vals) / step) * step, np.ceil(np.nanmax(vals) / step) * step


In [ ]:
# ---- Map 1: mean mindedness by jurisdiction --------------------------------------------
vmin, vmax = nice_range(state_m["mean_mindedness"])
if vmin == vmax:
    vmin, vmax = vmin - 5, vmax + 5
norm = Normalize(vmin, vmax)

fig = plt.figure(figsize=(11, 8.5))
ax = fig.add_axes([0.02, 0.15, 0.96, 0.7])
draw_us_map(ax, state_m["mean_mindedness"].to_dict(), norm,
            abbr_size=FONT * 1.05, value_size=FONT * 0.9, callout_size=FONT * 1.05)

fig.suptitle(
    f"Statutory Mindedness by Jurisdiction ({N_OFFENSES} offenses \u00d7 {N_JURIS} jurisdictions)\n"
    "mean mindedness index across each jurisdiction\u2019s statutes",
    fontsize=FONT * 2.2, fontweight="bold", y=0.955, va="top", linespacing=1.15)

cax = fig.add_axes([0.3, 0.1, 0.4, 0.022])
cb = fig.colorbar(ScalarMappable(norm=norm, cmap=MINDEDNESS_CMAP), cax=cax, orientation="horizontal")
cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=FONT * 1.3, length=2.5, width=0.6)
cb.set_label("Mean mindedness index (0 = no stated mens rea, 100 = purpose on every element)",
             fontsize=FONT * 1.3)

weights_txt = ", ".join(f"{k} {v:g}" for k, v in MINDEDNESS_WEIGHTS.items())
fig.text(0.5, 0.03,
         f"Mindedness per statute = average element weight \u00d7 100 ({weights_txt}). "
         "Color scale spans the observed range.",
         ha="center", fontsize=FONT * 1.1, color="#444444")
fig.savefig(OUT_DIR / "mindedness_map.png", dpi=300, facecolor="white")
fig.savefig(OUT_DIR / "mindedness_map.svg", facecolor="white")   # vector: stays sharp at any size
print(f"Saved {(OUT_DIR / 'mindedness_map').resolve()}.png and .svg")
plt.show()


In [ ]:
# ---- Map 2: one small map per offense (statute-level scores) ------------------------------
# Layout adapts to the number of offenses, like the bar graphs.
LEFT_IN, RIGHT_IN, TOP_IN, BOTTOM_IN, GAP_W = 0.35, 0.35, 1.0, 0.95, 0.1
REF_PANEL_W = 1.875           # panel width on the original 24-offense page (FONT looks right here)
_xmin, _ymin, _xmax, _ymax = GEO["lower48"].total_bounds
MAP_ASPECT = (_ymax - _ymin) * 1.04 / ((_xmax - _xmin) * 1.11)   # map height / width incl. padding

def _map_geometry(n, ncols, font, page_size):
    fig_w, fig_h = page_size
    nrows = int(np.ceil(n / ncols))
    gap_h = 0.38 + 0.38 * (font - 7) / 7               # room for up to 2-line panel titles
    ax_w = min((fig_w - LEFT_IN - RIGHT_IN - (ncols - 1) * GAP_W) / ncols,
               (fig_h - TOP_IN - BOTTOM_IN - (nrows - 1) * gap_h) / nrows / MAP_ASPECT)
    return dict(n=n, nrows=nrows, ncols=ncols, font=font, gap_h=gap_h, ax_w=ax_w,
                ax_h=ax_w * MAP_ASPECT, page_size=page_size)

def _map_fit(n, ncols, page_size):
    font = FONT
    for _ in range(5):                                 # font follows panel width; margins follow font
        g = _map_geometry(n, ncols, font, page_size)
        font = round(float(np.clip(FONT * g["ax_w"] / REF_PANEL_W, MIN_FONT, MAX_FONT)), 1)
    return _map_geometry(n, ncols, font, page_size)

def choose_map_layout(n, page_size=PAGE_SIZE, ncols=NCOLS):
    if ncols:
        return _map_fit(n, ncols, page_size)
    candidates = [g for g in (_map_fit(n, c, page_size) for c in range(1, n + 1)) if g["ax_w"] > 0]
    if not candidates:
        raise ValueError(f"{n} maps can't fit on a {page_size} page; set PANELS_PER_PAGE")
    return max(candidates, key=lambda g: g["ax_w"])

def plot_offense_maps(offenses, g, subtitle_suffix=""):
    fig_w, fig_h = g["page_size"]
    ncols, font, ax_w, ax_h, gap_h = g["ncols"], g["font"], g["ax_w"], g["ax_h"], g["gap_h"]
    nrows = int(np.ceil(len(offenses) / ncols))
    grid_w = ncols * ax_w + (ncols - 1) * GAP_W
    grid_h = nrows * ax_h + (nrows - 1) * gap_h
    x_left = (fig_w - grid_w) / 2
    y_top = TOP_IN + ((fig_h - TOP_IN - BOTTOM_IN) - grid_h) / 2
    norm = Normalize(0, 100)                           # common absolute scale for every offense

    fig = plt.figure(figsize=g["page_size"])
    title_size = font * 1.25
    wrap_chars = max(12, int(ax_w * 72 / (title_size * 0.55)))
    for i, off in enumerate(offenses):
        r, c = divmod(i, ncols)
        x = x_left + c * (ax_w + GAP_W)
        y = y_top + r * (ax_h + gap_h)
        ax = fig.add_axes([x / fig_w, 1 - (y + ax_h) / fig_h, ax_w / fig_w, ax_h / fig_h])
        vals = statute_m.loc[off, "mindedness"].to_dict()
        draw_us_map(ax, vals, norm, labels=False, callouts=["DC"], edge="#FFFFFF", edge_lw=0.15,
                    callout_size=font * 0.55)
        ax.set_title(textwrap.fill(off, wrap_chars), fontsize=title_size, fontweight="bold",
                     pad=2, linespacing=1.0, y=1.0)      # fixed y: stop auto-raising over insets

    fig.suptitle("Statute-Level Mindedness by Jurisdiction, Within Each Offense\n"
                 f"offenses ordered by amount of stated mens rea data{subtitle_suffix}",
                 fontsize=FONT * 1.6, fontweight="bold", y=1 - 0.3 / fig_h, va="top", linespacing=1.15)
    cax = fig.add_axes([0.3, 0.55 / fig_h, 0.4, 0.12 / fig_h])
    cb = fig.colorbar(ScalarMappable(norm=norm, cmap=MINDEDNESS_CMAP), cax=cax, orientation="horizontal")
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=FONT, length=2, width=0.5)
    cb.set_label("Mindedness index (0\u2013100)", fontsize=FONT)
    return fig

per_page = PANELS_PER_PAGE or len(order)
pages = [order[i:i + per_page] for i in range(0, len(order), per_page)]
layout = choose_map_layout(len(pages[0]))          # same grid on every page
print(f"{len(order)} offenses -> {len(pages)} page(s), {layout['ncols']} columns x {layout['nrows']} rows, "
      f"title text {layout['font'] * 1.25:.1f} pt")

for p, offs in enumerate(pages, start=1):
    multi = len(pages) > 1
    fig = plot_offense_maps(offs, layout, f" (page {p} of {len(pages)})" if multi else "")
    stem = "mindedness_by_offense_maps" + (f"_p{p}" if multi else "")
    fig.savefig(OUT_DIR / f"{stem}.png", dpi=300, facecolor="white")
    fig.savefig(OUT_DIR / f"{stem}.svg", facecolor="white")
    print(f"Saved {(OUT_DIR / stem).resolve()}.png and .svg")
    plt.show()
